<a href="https://colab.research.google.com/github/vonKleve/csci-e222-final-project/blob/master/ner/01-bioclinicalbert-baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vonKleve/csci-e222-final-project/blob/master/ner/01-bioclinicalbert-baseline.ipynb)

# Description

To load this from remote:
```
from transformers import pipeline

# It will download once and cache it for future use
ner_pipe = pipeline("ner", model="alexd063/bio-clinicalbert-finetuned-medmentions")
results = ner_pipe("Patient shows symptoms of acute appendicitis.")
print(results)
```

In [ ]:
%%capture
!pip install transformers datasets evaluate seqeval accelerate torch numpy
!pip install --upgrade transformers

In [ ]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from huggingface_hub import notebook_login, whoami

In [ ]:
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from datasets import disable_progress_bars
disable_progress_bars()

from huggingface_hub.utils import disable_progress_bars as hub_disable_progress_bars
hub_disable_progress_bars()

In [ ]:
PUSH_TO_HUB = True  # Set to False to skip login and hub push

if PUSH_TO_HUB:
    notebook_login()


In [ ]:
class Config:
    MODEL_ID = "emilyalsentzer/Bio_ClinicalBERT"
    DATASET_ID = "Ben10x/MedMentions-MTI881-NER"
    HUB_REPO_ID = "alexd063/bio-clinicalbert-finetuned-medmentions"

    MAX_LENGTH = 512
    BATCH_SIZE = 16
    EPOCHS = 20
    LEARNING_RATE = 2e-5
    WARMUP_RATIO = 0.10
    WEIGHT_DECAY = 0.01

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = Config()


In [ ]:
import time
notebook_start_time = time.time()


In [ ]:
# ==========================================
# 1. Load Dataset & Extract Labels
# ==========================================
dataset = load_dataset("Ben10x/MedMentions-MTI881-NER")

# the dataset stores tags as strings (e.g., 'B-T062', 'I-T062', 'O').
# we need to extract all unique tags to create our label mappings.
unique_tags = set()
for split in dataset.keys():
    for row in dataset[split]["ner_tags"]:
        unique_tags.update(row)

label_list = sorted(list(unique_tags))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# ==========================================
# 2. Tokenization & Label Alignment
# ==========================================
# tokenization
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_ID)

def tokenize_and_align_labels(examples):
    # tokenize the pre-split words
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=config.MAX_LENGTH
    )

    labels = []
    for i, label_sequence in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # special tokens (like [CLS] and [SEP]) map to None. we set their label to -100 so they are ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # only label the first token of a given word.
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label_sequence[word_idx]])
            # for subsequent tokens of the same word, also assign -100 (or you can assign the I- tag).
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# apply to train, validation, and test splits
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# ==========================================
# 3. Metrics
# ==========================================

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": np.round(results["overall_precision"], 3),
        "recall": np.round(results["overall_recall"], 3),
        "f1": np.round(results["overall_f1"], 3),
        "accuracy": np.round(results["overall_accuracy"], 3),
    }


In [ ]:

# ==========================================
# 4. Training
# ==========================================

training_args = TrainingArguments(
    output_dir="./bio-clinicalbert-medmentions",
    learning_rate=config.LEARNING_RATE,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    num_train_epochs=config.EPOCHS,
    weight_decay=config.WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    hub_model_id=config.HUB_REPO_ID if PUSH_TO_HUB else None
)

model = AutoModelForTokenClassification.from_pretrained(
    config.MODEL_ID,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

# save the final model locally
trainer.save_model("./bio-clinicalbert-medmentions-final")


In [ ]:
# ==========================================
# 5. Assessment on the Test Set
# ==========================================

test_results = trainer.evaluate(tokenized_datasets["test"])

print("--- Test Set Evaluation Results ---")
for key, value in test_results.items():
    # formatting key names for better readability (e.g., eval_f1 -> f1)
    metric_name = key.replace("eval_", "")
    print(f"{metric_name:10}: {value:.4f}")

# specific examples
# predictions, labels, metrics = trainer.predict(tokenized_datasets["test"])
# print("\nDetailed Test Metrics:", metrics)


In [ ]:
notebook_end_time = time.time()
elapsed = notebook_end_time - notebook_start_time
hours, rem = divmod(elapsed, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Total notebook execution time: {int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}")


In [ ]:

# ==========================================
# 7. Push to Hugging Face Hub
# ==========================================

if PUSH_TO_HUB:
    trainer.push_to_hub(
        commit_message="End of training",
        finetuned_from=config.MODEL_ID,
        dataset=config.DATASET_ID
    )
    tokenizer.push_to_hub(repo_id=config.HUB_REPO_ID)

    # sanity check: run inference from the remote model
    from transformers import pipeline

    ner_pipe = pipeline("ner", model=config.HUB_REPO_ID)
    results = ner_pipe("Patient shows symptoms of acute appendicitis.")
    print(results)
